# 📊 Predicción de Carga Operativa — Backoffice Call Center Financiero
**Modelo:** XGBoost Multiclase (Bajo / Medio / Alto)  
**Horizonte:** Próximos 7 días  
**País:** Honduras  

---
## Flujo del notebook
1. Carga y limpieza de datos  
2. Feature engineering (temporales, lags, rolling, feriados)  
3. Etiquetado de carga (bajo / medio / alto)  
4. Entrenamiento XGBoost con validación temporal  
5. Evaluación y visualización  
6. Predicción de los próximos 7 días  

## 0. Instalación de dependencias
Ejecuta esta celda solo la primera vez.

In [ ]:
# !pip install xgboost scikit-learn pandas numpy matplotlib seaborn openpyxl

## 1. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, accuracy_score
)

# Reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('✅ Librerías importadas correctamente')

## 2. Carga de datos

### 2.1 Datos históricos de solicitudes

> **Ajusta `RUTA_DATOS`** con la ruta a tu archivo CSV o Excel.

In [ ]:
# ─── CONFIGURA AQUÍ ───────────────────────────────────────────────────────────
RUTA_DATOS    = 'datos_solicitudes.csv'   # CSV o .xlsx
RUTA_FERIADOS = 'feriados_honduras.csv'   # columna 'fecha' con formato YYYY-MM-DD
# ──────────────────────────────────────────────────────────────────────────────

# Carga flexible CSV / Excel
def cargar_archivo(ruta):
    if ruta.endswith('.xlsx') or ruta.endswith('.xls'):
        return pd.read_excel(ruta)
    return pd.read_csv(ruta, encoding='utf-8-sig')

df_raw = cargar_archivo(RUTA_DATOS)

# Normalizar nombres de columnas
df_raw.columns = (
    df_raw.columns
    .str.strip()
    .str.upper()
    .str.replace(' ', '_')
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
)

print(f'📂 Filas cargadas : {len(df_raw):,}')
print(f'📋 Columnas       : {list(df_raw.columns)}')
df_raw.head(3)

In [ ]:
# ─── Mapea las columnas de tu archivo a los nombres estándar ──────────────────
# Si tus columnas ya tienen estos nombres exactos, no cambies nada.
# Si difieren, edita el diccionario.
MAPA_COLUMNAS = {
    'CREATED_DATE' : 'CREATED_DATE',
    'SOLICITUD_ID'  : 'SOLICITUD_ID',
    'GESTION_TYPE'  : 'GESTION_TYPE',
    'PRODUCT_TYPETC_OR_PR' : 'PRODUCT_TYPE',   # nombre normalizado
    'DELIQUENCY'    : 'DELIQUENCY',
    'CYCLE'         : 'CYCLE',
    'COUNTRY'       : 'COUNTRY',
}

df_raw.rename(columns=MAPA_COLUMNAS, inplace=True)
print('✅ Columnas renombradas:', list(df_raw.columns))

### 2.2 Carga de feriados

In [ ]:
df_feriados = cargar_archivo(RUTA_FERIADOS)
df_feriados.columns = df_feriados.columns.str.strip().str.lower()

# Se asume que el archivo de feriados tiene al menos una columna 'fecha'
df_feriados['fecha'] = pd.to_datetime(df_feriados['fecha'])
FERIADOS_SET = set(df_feriados['fecha'].dt.date)

print(f'📅 Feriados cargados: {len(FERIADOS_SET)}')
print(sorted(list(FERIADOS_SET))[:5], '...')

## 3. Limpieza y agregación diaria

In [ ]:
# Parsear fecha
df_raw['CREATED_DATE'] = pd.to_datetime(df_raw['CREATED_DATE'], dayfirst=False, errors='coerce')
df_raw['fecha'] = df_raw['CREATED_DATE'].dt.normalize()

# Eliminar registros sin fecha
n_before = len(df_raw)
df_raw.dropna(subset=['fecha'], inplace=True)
print(f'🗑️  Registros eliminados por fecha nula: {n_before - len(df_raw)}')

# Filtrar solo Honduras (si hay más países)
if df_raw['COUNTRY'].nunique() > 1:
    df_raw = df_raw[df_raw['COUNTRY'].str.upper().str.contains('HND|HONDURAS', na=False)]
    print(f'🌎 Registros filtrados para Honduras: {len(df_raw):,}')

# Agregación diaria
daily = (
    df_raw.groupby('fecha')
    .agg(
        total_solicitudes  = ('SOLICITUD_ID', 'count'),
        # Mix de producto: proporción TC
        pct_TC             = ('PRODUCT_TYPE', lambda x: (x.str.upper() == 'TC').mean()),
        # Mora promedio del día
        avg_deliquency     = ('DELIQUENCY', lambda x: pd.to_numeric(x, errors='coerce').mean()),
        # Ciclo promedio
        avg_cycle          = ('CYCLE', lambda x: pd.to_numeric(x, errors='coerce').mean()),
        # Tipos de gestión únicos
        n_gestiones_tipo   = ('GESTION_TYPE', 'nunique'),
    )
    .reset_index()
    .sort_values('fecha')
)

# Rellenar días sin registros (fines de semana / feriados sin actividad)
fecha_min = daily['fecha'].min()
fecha_max = daily['fecha'].max()
rango_completo = pd.date_range(fecha_min, fecha_max, freq='D')
daily = daily.set_index('fecha').reindex(rango_completo).rename_axis('fecha').reset_index()
daily['total_solicitudes'].fillna(0, inplace=True)

print(f'\n📊 Rango de datos: {fecha_min.date()} → {fecha_max.date()}')
print(f'📅 Total de días: {len(daily):,}')
daily.head()

## 4. Etiquetado de carga operativa

Usamos **percentiles** para definir los umbrales, lo que hace que las clases sean balanceadas independientemente del volumen real.

In [ ]:
# Percentiles sobre días con actividad real (total > 0)
dias_activos = daily[daily['total_solicitudes'] > 0]['total_solicitudes']

UMBRAL_BAJO  = dias_activos.quantile(0.33)
UMBRAL_ALTO  = dias_activos.quantile(0.67)

print(f'📊 Distribución de solicitudes diarias:')
print(f'   Mínimo  : {dias_activos.min():.0f}')
print(f'   P33     : {UMBRAL_BAJO:.0f}  ← umbral BAJO/MEDIO')
print(f'   Mediana : {dias_activos.median():.0f}')
print(f'   P67     : {UMBRAL_ALTO:.0f}  ← umbral MEDIO/ALTO')
print(f'   Máximo  : {dias_activos.max():.0f}')

def etiquetar_carga(n):
    if n <= UMBRAL_BAJO:
        return 'BAJO'
    elif n <= UMBRAL_ALTO:
        return 'MEDIO'
    else:
        return 'ALTO'

daily['carga'] = daily['total_solicitudes'].apply(etiquetar_carga)

# Distribución de clases
print('\n🏷️  Distribución de clases:')
print(daily['carga'].value_counts())

In [ ]:
# Visualización de la serie histórica con etiquetas
COLORES = {'BAJO': '#2ecc71', 'MEDIO': '#f39c12', 'ALTO': '#e74c3c'}

fig, ax = plt.subplots(figsize=(16, 4))
for etiqueta, color in COLORES.items():
    mask = daily['carga'] == etiqueta
    ax.scatter(daily.loc[mask, 'fecha'], daily.loc[mask, 'total_solicitudes'],
               label=etiqueta, color=color, s=18, alpha=0.7, zorder=3)

ax.plot(daily['fecha'], daily['total_solicitudes'], color='#bdc3c7', linewidth=0.5, zorder=1)
ax.axhline(UMBRAL_BAJO, linestyle='--', color='#2ecc71', alpha=0.4, linewidth=1)
ax.axhline(UMBRAL_ALTO, linestyle='--', color='#e74c3c', alpha=0.4, linewidth=1)
ax.set_title('Serie histórica de solicitudes diarias — Nivel de carga', fontsize=13, pad=12)
ax.set_xlabel('Fecha')
ax.set_ylabel('Total solicitudes')
ax.legend(title='Carga')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
def build_features(df, feriados_set):
    d = df.copy()

    # ── Temporales básicas ────────────────────────────────────────────────────
    d['dia_semana']     = d['fecha'].dt.dayofweek          # 0=Lunes, 6=Domingo
    d['dia_mes']        = d['fecha'].dt.day
    d['semana_anio']    = d['fecha'].dt.isocalendar().week.astype(int)
    d['mes']            = d['fecha'].dt.month
    d['trimestre']      = d['fecha'].dt.quarter
    d['anio']           = d['fecha'].dt.year
    d['es_fin_semana']  = (d['dia_semana'] >= 5).astype(int)
    d['es_lunes']       = (d['dia_semana'] == 0).astype(int)
    d['es_viernes']     = (d['dia_semana'] == 4).astype(int)
    d['es_inicio_mes']  = (d['dia_mes'] <= 5).astype(int)
    d['es_fin_mes']     = (d['dia_mes'] >= 25).astype(int)
    d['es_inicio_trim'] = (d['mes'].isin([1, 4, 7, 10]) & (d['dia_mes'] <= 5)).astype(int)

    # ── Feriados ─────────────────────────────────────────────────────────────
    d['es_feriado']          = d['fecha'].dt.date.isin(feriados_set).astype(int)
    d['pre_feriado']         = d['fecha'].apply(
        lambda f: int((f + pd.Timedelta(days=1)).date() in feriados_set))
    d['post_feriado']        = d['fecha'].apply(
        lambda f: int((f - pd.Timedelta(days=1)).date() in feriados_set))

    # ── Lags (días anteriores) ────────────────────────────────────────────────
    for lag in [1, 2, 3, 5, 7, 14]:
        d[f'lag_{lag}d'] = d['total_solicitudes'].shift(lag)

    # ── Rolling statistics ────────────────────────────────────────────────────
    for ventana in [7, 14, 30]:
        d[f'roll_mean_{ventana}d'] = (
            d['total_solicitudes'].shift(1).rolling(ventana).mean())
        d[f'roll_std_{ventana}d']  = (
            d['total_solicitudes'].shift(1).rolling(ventana).std())
        d[f'roll_max_{ventana}d']  = (
            d['total_solicitudes'].shift(1).rolling(ventana).max())

    # ── Tendencia semanal (mismo día semana anterior) ─────────────────────────
    d['mismo_dia_semana_pasada'] = d['total_solicitudes'].shift(7)

    # ── Variables contextuales del día ───────────────────────────────────────
    d['pct_TC'].fillna(d['pct_TC'].median(), inplace=True)
    d['avg_deliquency'].fillna(d['avg_deliquency'].median(), inplace=True)
    d['avg_cycle'].fillna(d['avg_cycle'].median(), inplace=True)
    d['n_gestiones_tipo'].fillna(0, inplace=True)

    return d

daily = build_features(daily, FERIADOS_SET)

# Eliminar filas con NaN en lags (primeras 2 semanas)
FEATURE_COLS = [c for c in daily.columns if c not in
    ['fecha', 'total_solicitudes', 'carga']]

daily_clean = daily.dropna(subset=FEATURE_COLS).copy()
print(f'✅ Features construidas: {len(FEATURE_COLS)} variables')
print(f'📊 Registros listos para modelar: {len(daily_clean):,}')
FEATURE_COLS

## 6. División temporal Train / Validation

> **Importante:** En series de tiempo NO se hace split aleatorio.  
> Entrenamos con todo hasta hace 60 días y validamos con los últimos 60 días.

In [ ]:
DIAS_VALIDACION = 60   # Puedes ajustar a 30 o 90

fecha_corte = daily_clean['fecha'].max() - pd.Timedelta(days=DIAS_VALIDACION)

train = daily_clean[daily_clean['fecha'] <= fecha_corte].copy()
val   = daily_clean[daily_clean['fecha'] >  fecha_corte].copy()

print(f'📅 Corte de validación : {fecha_corte.date()}')
print(f'🏋️  Train : {len(train):,} días  ({train["fecha"].min().date()} → {train["fecha"].max().date()})')
print(f'🔍 Val   : {len(val):,} días  ({val["fecha"].min().date()} → {val["fecha"].max().date()})')

# Encode target
le = LabelEncoder()
le.fit(['BAJO', 'MEDIO', 'ALTO'])
print(f'\n🏷️  Clases codificadas: {dict(zip(le.classes_, le.transform(le.classes_)))}')

X_train = train[FEATURE_COLS]
y_train = le.transform(train['carga'])
X_val   = val[FEATURE_COLS]
y_val   = le.transform(val['carga'])

## 7. Entrenamiento XGBoost

In [ ]:
modelo = XGBClassifier(
    objective         = 'multi:softprob',
    num_class         = 3,
    n_estimators      = 400,
    learning_rate     = 0.05,
    max_depth         = 5,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 3,
    gamma             = 0.1,
    use_label_encoder = False,
    eval_metric       = 'mlogloss',
    early_stopping_rounds = 30,
    random_state      = RANDOM_STATE,
    verbosity         = 0,
)

modelo.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

print(f'\n✅ Mejor iteración: {modelo.best_iteration}')

## 8. Evaluación del modelo

In [ ]:
y_pred = modelo.predict(X_val)
y_pred_labels = le.inverse_transform(y_pred)
y_val_labels  = le.inverse_transform(y_val)

acc = accuracy_score(y_val, y_pred)
print(f'🎯 Accuracy en validación: {acc:.2%}\n')
print(classification_report(y_val_labels, y_pred_labels,
                             target_names=['ALTO', 'BAJO', 'MEDIO']))

In [ ]:
# Matriz de confusión
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz
cm = confusion_matrix(y_val_labels, y_pred_labels, labels=['BAJO', 'MEDIO', 'ALTO'])
disp = ConfusionMatrixDisplay(cm, display_labels=['BAJO', 'MEDIO', 'ALTO'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Matriz de Confusión')

# Importancia de features (top 20)
importancias = pd.Series(modelo.feature_importances_, index=FEATURE_COLS)
top20 = importancias.nlargest(20).sort_values()
top20.plot(kind='barh', ax=axes[1], color='#3498db')
axes[1].set_title('Top 20 Features por importancia')
axes[1].set_xlabel('F-score')

plt.tight_layout()
plt.show()

In [ ]:
# Comparación real vs predicho en el período de validación
df_resultado = val[['fecha', 'total_solicitudes', 'carga']].copy()
df_resultado['pred_carga'] = y_pred_labels
df_resultado['correcto']   = df_resultado['carga'] == df_resultado['pred_carga']

fig, ax = plt.subplots(figsize=(16, 4))
for etiqueta, color in COLORES.items():
    mask = df_resultado['pred_carga'] == etiqueta
    ax.scatter(df_resultado.loc[mask, 'fecha'],
               df_resultado.loc[mask, 'total_solicitudes'],
               label=f'Pred: {etiqueta}', color=color, s=60,
               marker='^', zorder=4, alpha=0.8)

ax.plot(df_resultado['fecha'], df_resultado['total_solicitudes'],
        color='#95a5a6', linewidth=1, zorder=1)
ax.set_title('Período de validación — Real (línea) vs Predicho (triángulos)', fontsize=12)
ax.set_xlabel('Fecha')
ax.set_ylabel('Solicitudes')
ax.legend(title='Predicción')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
plt.tight_layout()
plt.show()

## 9. Predicción de los próximos 7 días

Usamos los últimos 30 días como ventana de contexto para calcular lags y rolling stats para cada día futuro.

In [ ]:
def predecir_proximos_dias(modelo, daily_clean, feriados_set, le, feature_cols, n_dias=7):
    """
    Predicción iterativa de n_dias hacia adelante.
    Cada día predicho se agrega al historial para que los lags sean consistentes.
    """
    # Contexto: últimos 60 días reales como punto de partida
    contexto = daily_clean[['fecha', 'total_solicitudes',
                             'pct_TC', 'avg_deliquency',
                             'avg_cycle', 'n_gestiones_tipo']].copy()

    ultima_fecha = contexto['fecha'].max()
    resultados   = []

    for i in range(1, n_dias + 1):
        fecha_pred = ultima_fecha + pd.Timedelta(days=i)

        # Placeholder: el valor desconocido se estima con la media rolling
        # (se usará solo para los contextos de días futuros subsiguientes)
        nueva_fila = {
            'fecha'           : fecha_pred,
            'total_solicitudes': np.nan,  # desconocido
            'pct_TC'          : contexto['pct_TC'].iloc[-30:].mean(),
            'avg_deliquency'  : contexto['avg_deliquency'].iloc[-30:].mean(),
            'avg_cycle'       : contexto['avg_cycle'].iloc[-30:].mean(),
            'n_gestiones_tipo': contexto['n_gestiones_tipo'].iloc[-7:].mean(),
        }
        contexto = pd.concat(
            [contexto, pd.DataFrame([nueva_fila])], ignore_index=True)

        # Reconstruir features sobre el contexto ampliado
        ctx_features = build_features(contexto, feriados_set)
        fila_pred    = ctx_features[ctx_features['fecha'] == fecha_pred]

        if fila_pred.empty or fila_pred[feature_cols].isnull().any(axis=1).all():
            resultados.append({'fecha': fecha_pred, 'pred_carga': 'SIN DATOS',
                                'prob_bajo': 0, 'prob_medio': 0, 'prob_alto': 0})
            continue

        X_pred     = fila_pred[feature_cols].fillna(method='ffill')
        proba      = modelo.predict_proba(X_pred)[0]
        clase_idx  = np.argmax(proba)
        clase_pred = le.inverse_transform([clase_idx])[0]

        # Actualizar total_solicitudes con la media del nivel predicho
        # (para que los lags de días subsiguientes sean razonables)
        if clase_pred == 'ALTO':
            estimado = daily_clean[daily_clean['carga'] == 'ALTO']['total_solicitudes'].mean()
        elif clase_pred == 'MEDIO':
            estimado = daily_clean[daily_clean['carga'] == 'MEDIO']['total_solicitudes'].mean()
        else:
            estimado = daily_clean[daily_clean['carga'] == 'BAJO']['total_solicitudes'].mean()

        contexto.loc[contexto['fecha'] == fecha_pred, 'total_solicitudes'] = estimado

        resultados.append({
            'fecha'      : fecha_pred,
            'dia_semana' : fecha_pred.strftime('%A'),
            'pred_carga' : clase_pred,
            'prob_bajo'  : round(proba[le.transform(['BAJO'])[0]], 3),
            'prob_medio' : round(proba[le.transform(['MEDIO'])[0]], 3),
            'prob_alto'  : round(proba[le.transform(['ALTO'])[0]], 3),
        })

    return pd.DataFrame(resultados)


df_pred = predecir_proximos_dias(
    modelo, daily_clean, FERIADOS_SET, le, FEATURE_COLS, n_dias=7
)

df_pred

In [ ]:
# ── Visualización del pronóstico ──────────────────────────────────────────────
COLORES_HEX = {'BAJO': '#2ecc71', 'MEDIO': '#f39c12', 'ALTO': '#e74c3c',
               'SIN DATOS': '#bdc3c7'}

fig, ax = plt.subplots(figsize=(12, 5))

bars = ax.bar(
    df_pred['fecha'].dt.strftime('%a\n%d %b'),
    df_pred['prob_alto'],
    color=[COLORES_HEX.get(c, '#bdc3c7') for c in df_pred['pred_carga']],
    edgecolor='white', linewidth=1.5, alpha=0.85
)

# Probabilidades apiladas
ax2 = ax.twinx()
ax2.bar(
    df_pred['fecha'].dt.strftime('%a\n%d %b'),
    df_pred['prob_medio'],
    bottom=df_pred['prob_alto'],
    color='#f39c12', alpha=0.4, edgecolor='white', linewidth=1
)
ax2.bar(
    df_pred['fecha'].dt.strftime('%a\n%d %b'),
    df_pred['prob_bajo'],
    bottom=df_pred['prob_alto'] + df_pred['prob_medio'],
    color='#2ecc71', alpha=0.4, edgecolor='white', linewidth=1
)
ax2.set_ylim(0, 1)
ax2.set_visible(False)

# Etiquetas
for bar, (_, row) in zip(bars, df_pred.iterrows()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{row['pred_carga']}\n{row['prob_alto']:.0%}",
        ha='center', va='bottom', fontsize=9, fontweight='bold',
        color=COLORES_HEX.get(row['pred_carga'], '#555')
    )

ax.set_ylim(0, 1.25)
ax.set_ylabel('Probabilidad de carga ALTA')
ax.set_title('📅 Pronóstico de carga — Próximos 7 días', fontsize=13, pad=12)
ax.set_xlabel('Día')

from matplotlib.patches import Patch
leyenda = [Patch(facecolor=v, label=k) for k, v in COLORES_HEX.items() if k != 'SIN DATOS']
ax.legend(handles=leyenda, title='Nivel predicho', loc='upper right')

plt.tight_layout()
plt.show()

print('\n📋 Detalle del pronóstico:')
display(df_pred[['fecha', 'dia_semana', 'pred_carga', 'prob_bajo', 'prob_medio', 'prob_alto']])

## 10. Exportar resultados

In [ ]:
# Guardar pronóstico
df_pred.to_csv('pronostico_7dias.csv', index=False)

# Guardar histórico con etiquetas y predicciones de validación
df_resultado.to_csv('historico_etiquetado.csv', index=False)

# Guardar modelo
modelo.save_model('modelo_xgb_carga.json')

print('✅ Archivos exportados:')
print('   pronostico_7dias.csv')
print('   historico_etiquetado.csv')
print('   modelo_xgb_carga.json')

---
## 📝 Próximos pasos recomendados

| Paso | Descripción |
|------|-------------|
| **Ajuste de umbrales** | Redefine BAJO/MEDIO/ALTO según tu capacidad real de agentes por turno |
| **Hyperparameter tuning** | Usa `Optuna` o `GridSearchCV` con `TimeSeriesSplit` para optimizar XGBoost |
| **Variables externas** | Agrega campañas activas, vencimientos de cartera o eventos del calendario bancario |
| **Monitoreo de drift** | Compara la distribución de features semanalmente con `evidently` |
| **Reentrenamiento automático** | Programa un script que re-entrene el modelo cada mes con datos frescos |
| **Dashboard operativo** | Conecta `pronostico_7dias.csv` a Power BI / Tableau para visualización diaria |